In [ ]:
from fastapi import FastAPI, HTTPException
import sqlite3
import os
from dotenv import load_dotenv
from starlette.middleware.cors import CORSMiddleware
import logging
import json


# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Load secrets
load_dotenv()


# Initialize FastAPI
app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

# SQLite connection
def get_db_connection():
    try:
        conn = sqlite3.connect(os.getenv('SQLITE_DB_PATH', 'rumie.db'))
        conn.row_factory = sqlite3.Row
        logger.info("SQLite connection established")
        return conn
    except sqlite3.Error as e:
        logger.error(f"SQLite error: {e}")
        raise

# Mock Redis (in-memory dict for Day 1)
mock_redis = {}

# Mock integration data (Google Workspace: Gmail/Calendar, Notion)
mock_integrations = {
    "1": {
        "gmail": [{"subject": "Urgent: Project Update", "sender": "bob@company.com", "urgent": True}],
        "calendar": [{"title": "Team Meeting", "time": "2025-09-24 15:00", "duration": "1h"}],
        "notion": [{"task": "Finish MVP Plan", "due": "2025-09-24", "priority": "High"}]
    }
}

# Initialize DB schema
try:
    with get_db_connection() as conn:
        conn.execute('''
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT UNIQUE NOT NULL,
                mode_pref TEXT DEFAULT 'work',
                personality_trait TEXT
            )
        ''')
        conn.execute('INSERT OR IGNORE INTO users (email, mode_pref, personality_trait) VALUES (?, ?, ?)',
                    ('test@rumie.ai', 'work', 'efficient_organizer'))
        conn.commit()
        logger.info("SQLite schema initialized")
except sqlite3.Error as e:
    logger.error(f"Schema init error: {e}")
    raise

@app.get("/users/{user_id}")
async def get_user(user_id: int):
    try:
        with get_db_connection() as conn:
            user = conn.execute('SELECT * FROM users WHERE id = ?', (user_id,)).fetchone()
            if not user:
                logger.warning(f"User {user_id} not found")
                raise HTTPException(status_code=404, detail="User not found")
            user_dict = dict(user)
            mock_redis[f"user:{user_id}"] = json.dumps(user_dict)
            logger.info(f"User {user_id} fetched and cached")
            return user_dict
    except Exception as e:
        logger.error(f"Get user error: {e}")
        raise HTTPException(status_code=500, detail=f"Error: {str(e)}")

@app.get("/test_redis")
async def test_redis():
    try:
        mock_redis["test_key"] = "test_value"
        value = mock_redis.get("test_key")
        logger.info("Mock Redis test successful")
        return {"value": value}
    except Exception as e:
        logger.error(f"Mock Redis error: {e}")
        raise HTTPException(status_code=500, detail=f"Mock Redis error: {str(e)}")

@app.get("/mode_response/{user_id}")
async def mode_response(user_id: int):
    try:
        with get_db_connection() as conn:
            user = conn.execute('SELECT mode_pref FROM users WHERE id = ?', (user_id,)).fetchone()
            if not user:
                logger.warning(f"User {user_id} not found")
                raise HTTPException(status_code=404, detail="User not found")
            mode = user['mode_pref']
            integrations = mock_integrations.get(str(user_id), {"gmail": [], "calendar": [], "notion": []})
            gmail = integrations["gmail"]
            calendar = integrations["calendar"]
            notion = integrations["notion"]
            response = ""
            if mode == 'work':
                if gmail:
                    response += f"Urgent email from {gmail[0]['sender']} ('{gmail[0]['subject']}'). Draft a reply?"
                if calendar:
                    response += f" Your {calendar[0]['title']} is at {calendar[0]['time'][:16]}. Prep needed?"
                if notion:
                    response += f" Notion task '{notion[0]['task']}' (due {notion[0]['due'][:10]}) is {notion[0]['priority']} priority."
            else:  # relax
                if calendar:
                    response += f"Busy day with a {calendar[0]['title']} at {calendar[0]['time'][:16]}. Try a 5-min breathing exercise to stay calm?"
                if gmail:
                    response += f" That email from {gmail[0]['sender']} can wait—let’s focus on your well-being."
                if notion:
                    response += f" Your Notion task '{notion[0]['task']}' is due {notion[0]['due'][:10]}. We’ll tackle it calmly."
                if not response:
                    response = "All clear—time to relax!"

                # Manually enhanced response for mindfulness
                enhanced_response = "Feeling a little overwhelmed? Let's take a moment to breathe. Find a comfortable position, close your eyes if you like, and just focus on your breath for a few minutes. Inhale deeply, exhale slowly. Everything else can wait."

                # Combine the original response with the mindfulness suggestion
                if response:
                     response = f"{response.strip()} {enhanced_response}"
                else:
                     response = enhanced_response


            if not response:
                response = "No tasks or emails to worry about—let’s plan your day!" if mode == 'work' else "All clear—time to relax!"


            logger.info(f"Mode response for user {user_id}: {mode}")
            return {"user_id": user_id, "mode": mode, "response": response.strip()}
    except Exception as e:
        logger.error(f"Mode response error: {e}")
        raise HTTPException(status_code=500, detail=f"Error: {str(e)}")

@app.get("/chat/context/{user_id}")
async def get_chat_context(user_id: int):
    try:
        context = mock_redis.get(f"chat:{user_id}")
        if not context:
            logger.warning(f"Chat context for user {user_id} not found")
            return {"chat_history": []}
        logger.info(f"Chat context for user {user_id} fetched")
        return {"chat_history": json.loads(context)}
    except Exception as e:
        logger.error(f"Get chat context error: {e}")
        raise HTTPException(status_code=500, detail=f"Error: {str(e)}")

@app.post("/chat/context/{user_id}")
async def set_chat_context(user_id: int, chat_history: list):
    try:
        mock_redis[f"chat:{user_id}"] = json.dumps(chat_history)
        logger.info(f"Chat context for user {user_id} updated")
        return {"status": "updated"}
    except Exception as e:
        logger.error(f"Set chat context error: {e}")
        raise HTTPException(status_code=500, detail=f"Error: {str(e)}")

INFO:__main__:SQLite connection established
INFO:__main__:SQLite schema initialized


In [3]:
!pip install fastapi uvicorn redis tenacity python-dotenv pytest redis


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: C:\Users\asusa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
!pip install pyngrok
# Test FastAPI endpoints
import requests
import json

# Load ngrok URL from environment variable
import os
from dotenv import load_dotenv
load_dotenv()

# Get ngrok URL from environment variable
public_url = os.getenv('NGROK_URL', "https://bernadette-uncondensational-mckenzie.ngrok-free.dev/")
print(f"Using ngrok URL: {public_url}")

def test_endpoint(endpoint_name, url):
    """Helper function to test endpoints with proper error handling"""
    try:
        print(f"\nTesting {endpoint_name}:")
        response = requests.get(url, timeout=10)
        print(f"Status Code: {response.status_code}")
        print(f"Response Headers: {dict(response.headers)}")
        print(f"Response Text: {response.text[:500]}...")  # First 500 chars
        
        if response.status_code == 200:
            try:
                json_data = response.json()
                print(f"JSON Response: {json_data}")
                return json_data
            except json.JSONDecodeError as e:
                print(f"JSON Decode Error: {e}")
                print(f"Raw response: {response.text}")
                return None
        else:
            print(f"HTTP Error: {response.status_code}")
            return None
            
    except requests.RequestException as e:
        print(f"Request error: {e}")
        return None

# Test endpoints with better error handling
test_endpoint("/users/1", f"{public_url}/users/1")
test_endpoint("/mode_response/1 (Work Mode)", f"{public_url}/mode_response/1")
test_endpoint("/test_redis", f"{public_url}/test_redis")
test_endpoint("/chat/context/1", f"{public_url}/chat/context/1")

# Update user to Relax Mode
import sqlite3

# Use absolute path to ensure we're connecting to the correct database
db_path = os.path.join(os.getcwd(), 'rumie.db')
print(f"\nConnecting to database at: {db_path}")

try:
    with sqlite3.connect(db_path) as conn:
        conn.execute("UPDATE users SET mode_pref = 'relax' WHERE id = 1")
        conn.commit()
    print("User 1 mode updated to relax")
except sqlite3.Error as e:
    print(f"Database error: {e}")

# Retest /mode_response/1 (Relax Mode)
test_endpoint("/mode_response/1 (Relax Mode)", f"{public_url}/mode_response/1")

Using ngrok URL: https://bernadette-uncondensational-mckenzie.ngrok-free.dev/

Testing /users/1:



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: C:\Users\asusa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Status Code: 400
Response Headers: {'Connection': 'close', 'Content-Type': 'text/html', 'Ngrok-Error-Code': 'ERR_NGROK_8012', 'Referrer-Policy': 'no-referrer', 'Date': 'Tue, 23 Sep 2025 14:27:57 GMT', 'Transfer-Encoding': 'chunked'}
Response Text: <!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <link rel="preload" href="https://cdn.ngrok.com/static/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://cdn.ngrok.com/static/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://cdn.ngrok.com/static/fonts/euclid-square/EuclidSquare-Me...
HTTP Error: 400

Testing /mode_response/1 (Work Mode):
Status Code: 400
Response Headers: {'Connection': 'close', 'Content-Type': 'text/html', 'Ngrok-Error-Code': 'ERR_NGROK_8012', 'Referrer-Policy': 'no-referrer', 'Date': 'Tue, 23 Sep 2025 14:27:5

In [13]:
from pyngrok import ngrok
import os
import subprocess
import time
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get token from .env file
NGROK_TOKEN = os.getenv("NGROK")

if not NGROK_TOKEN:
    raise ValueError("NGROK token not found in .env file. Please add NGROK=your_token_here to your .env file")

# Set it for pyngrok
ngrok.set_auth_token(NGROK_TOKEN)

def kill_all_ngrok_processes():
    """Kill all ngrok processes to ensure clean state"""
    try:
        # Kill ngrok processes on Windows
        subprocess.run(['taskkill', '/f', '/im', 'ngrok.exe'], 
                      capture_output=True, text=True)
        print("🔄 Killed all ngrok processes")
        time.sleep(2)  # Wait for processes to fully terminate
    except Exception as e:
        print(f"Error killing ngrok processes: {e}")

def create_ngrok_tunnel():
    """Create a new ngrok tunnel with retry logic"""
    max_retries = 3
    
    for attempt in range(max_retries):
        try:
            print(f"🔄 Attempt {attempt + 1} to create tunnel...")
            
            # Try to create tunnel
            public_url = ngrok.connect("8000")
            print("🚀 New tunnel created!")
            print("🌐 Public URL:", public_url.public_url)
            
            # Save the URL to environment for other cells to use
            if public_url.public_url:
                os.environ['NGROK_URL'] = public_url.public_url
                return public_url
                
        except Exception as e:
            print(f"❌ Attempt {attempt + 1} failed: {e}")
            
            if "already online" in str(e) or "ERR_NGROK_334" in str(e):
                print("🔄 Detected existing tunnel conflict, killing all ngrok processes...")
                kill_all_ngrok_processes()
                
                if attempt < max_retries - 1:
                    print("⏳ Waiting before retry...")
                    time.sleep(3)
                    continue
            else:
                print(f"❌ Unexpected error: {e}")
                break
    
    # If all attempts failed, try to use existing tunnel
    print("🔄 All creation attempts failed, trying to use existing tunnel...")
    try:
        tunnels = ngrok.get_tunnels()
        if tunnels:
            public_url = tunnels[0]
            print("🔄 Using existing tunnel:")
            print("🌐 Public URL:", public_url.public_url)
            if public_url.public_url:
                os.environ['NGROK_URL'] = public_url.public_url
            return public_url
        else:
            print("❌ No tunnels available")
            return None
    except Exception as e:
        print(f"❌ Error getting existing tunnel: {e}")
        return None

# Main execution
print("🚀 Starting ngrok tunnel setup...")

# First, try to kill any existing ngrok processes
kill_all_ngrok_processes()

# Create the tunnel
tunnel = create_ngrok_tunnel()

if tunnel:
    print("✅ Tunnel setup complete!")
    print(f"🌐 Your FastAPI server should be accessible at: {tunnel.public_url}")
else:
    print("❌ Failed to create or find a working tunnel")
    print("💡 Try manually running: ngrok http 8000")


INFO:pyngrok.process:Updating authtoken for default "config_path" of "ngrok_path": C:\Users\asusa\AppData\Local\ngrok\ngrok.exe


🚀 Starting ngrok tunnel setup...
🔄 Killed all ngrok processes


INFO:pyngrok.ngrok:Opening tunnel named: http-8000-4835efe1-fd3b-4ec5-96e2-668cc22a54ed
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:16+0530 lvl=info msg="no configuration paths supplied"
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:16+0530 lvl=info msg="using configuration at default config path" path=C:\\Users\\asusa\\AppData\\Local/ngrok/ngrok.yml
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:16+0530 lvl=info msg="open config file" path=C:\\Users\\asusa\\AppData\\Local\\ngrok\\ngrok.yml err=nil
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:16+0530 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]


🔄 Attempt 1 to create tunnel...


INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:17+0530 lvl=info msg="client session established" obj=tunnels.session
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:17+0530 lvl=info msg="tunnel session started" obj=tunnels.session
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:17+0530 lvl=info msg=start pg=/api/tunnels id=020deb57f6e9bb84
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:17+0530 lvl=info msg=end pg=/api/tunnels id=020deb57f6e9bb84 status=200 dur=524.3Âµs
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:17+0530 lvl=info msg=start pg=/api/tunnels id=a7c15068970a7c42
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:17+0530 lvl=info msg=end pg=/api/tunnels id=a7c15068970a7c42 status=200 dur=0s
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:17+0530 lvl=info msg=start pg=/api/tunnels id=b7f67ab336024e7b
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:17+0530 lvl=info msg=end pg=/api/tunnels id=b7f67ab336024e7b status=200 dur=506.3Âµs
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:17+0530 lvl=

...

INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg="no configuration paths supplied"
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg="using configuration at default config path" path=C:\\Users\\asusa\\AppData\\Local/ngrok/ngrok.yml
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg="open config file" path=C:\\Users\\asusa\\AppData\\Local\\ngrok\\ngrok.yml err=nil
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]


🔄 All creation attempts failed, trying to use existing tunnel...


INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg="client session established" obj=tunnels.session
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg="tunnel session started" obj=tunnels.session
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg=start pg=/api/tunnels id=a28fff4f615aa027
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg=end pg=/api/tunnels id=a28fff4f615aa027 status=200 dur=783.2Âµs
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg=start pg=/api/tunnels id=fea50a49ef7984e4
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg=end pg=/api/tunnels id=fea50a49ef7984e4 status=200 dur=0s
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg=start pg=/api/tunnels id=5737aedd508a0381
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg=end pg=/api/tunnels id=5737aedd508a0381 status=200 dur=0s


❌ No tunnels available
❌ Failed to create or find a working tunnel
💡 Try manually running: ngrok http 8000


INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg=start pg=/api/tunnels id=def550f27130fdd9
INFO:pyngrok.process.ngrok:t=2025-09-23T20:06:31+0530 lvl=info msg=end pg=/api/tunnels id=def550f27130fdd9 status=200 dur=0s
